# Token Pruning and Merging

## Learning Objectives
1. Understand attention-rollout and magnitude-based token importance scoring
2. Implement hard token pruning that reduces sequence length
3. Apply Token Merging (ToMe) bipartite soft-matching
4. Compare pruning vs merging on throughput and quality trade-offs

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## Level 1: Basic Token Pruning

In [ ]:
# Simulate 128 tokens with 64-dim embeddings
tokens = np.random.randn(128, 64)
# Simulate attention weights [layers, heads, seq, seq]
attn_weights = np.stack([
    np.stack([np.random.dirichlet(np.ones(128), size=128) for _ in range(12)])
    for _ in range(6)
])  # [6, 12, 128, 128]

def compute_attention_rollout(attn_weights):
    """Propagate attention across layers to get global token importance."""
    B = np.eye(attn_weights.shape[-1])
    for layer in attn_weights:
        avg = layer.mean(0)   # average over heads -> [seq, seq]
        B = avg @ B
    return B[0]  # importance from CLS token perspective

def magnitude_importance(tokens):
    """L2 norm of each token vector as proxy for information content."""
    return np.linalg.norm(tokens, axis=-1)

scores_rollout = compute_attention_rollout(attn_weights)
scores_mag     = magnitude_importance(tokens)

def prune_tokens(tokens, scores, keep_ratio=0.5):
    k = max(1, int(len(tokens) * keep_ratio))
    top_idx = np.argsort(scores)[-k:]
    return tokens[np.sort(top_idx)], top_idx

kept_50_r, idx_50 = prune_tokens(tokens, scores_rollout, 0.50)
kept_25_r, idx_25 = prune_tokens(tokens, scores_rollout, 0.25)
kept_50_m, _      = prune_tokens(tokens, scores_mag,     0.50)

print(f"Original tokens shape : {tokens.shape}")
print(f"Rollout keep-50%      : {kept_50_r.shape}  ({(1-len(kept_50_r)/len(tokens)):.0%} pruned)")
print(f"Rollout keep-25%      : {kept_25_r.shape}  ({(1-len(kept_25_r)/len(tokens)):.0%} pruned)")
print(f"Magnitude keep-50%   : {kept_50_m.shape}  ({(1-len(kept_50_m)/len(tokens)):.0%} pruned)")

# FLOPs savings: attention is O(n^2)
def attn_flops(seq_len, dim=64):
    return 2 * seq_len * seq_len * dim

print("\nFLOPs vs keep ratio:")
for ratio in [1.0, 0.75, 0.50, 0.25]:
    n = int(128 * ratio)
    f = attn_flops(n)
    print(f"  keep {ratio:.0%}: seq={n:3d}, FLOPs={f:>8,}  ({f/attn_flops(128):.1%} of baseline)")

## Level 2: Advanced Token Merging (ToMe)

In [ ]:
class ToMeLayer(nn.Module):
    """Token Merging: reduce seq_len by r merges per forward pass (Bolya et al. 2022)."""
    def __init__(self, dim: int, r: int = 16):
        super().__init__()
        self.r = r
        self.dim = dim

    def bipartite_soft_matching(self, x: torch.Tensor):
        B, N, C = x.shape
        half = N // 2
        src, dst = x[:, :half], x[:, half:]

        # Cosine similarity between src and dst tokens
        src_n = nn.functional.normalize(src, dim=-1)
        dst_n = nn.functional.normalize(dst, dim=-1)
        sim = torch.bmm(src_n, dst_n.transpose(1, 2))  # [B, half, half]

        best_match = sim.argmax(dim=-1)   # each src -> best dst
        best_sim   = sim.max(dim=-1).values

        # Select the r most-similar pairs to merge
        r = min(self.r, half)
        _, top_r_idx = best_sim.topk(r, dim=-1)
        return src, dst, best_match, top_r_idx

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        src, dst, match_idx, merge_idx = self.bipartite_soft_matching(x)

        merged_dst = dst.clone()
        for b in range(B):
            for i in merge_idx[b]:
                j = match_idx[b, i]
                merged_dst[b, j] = (merged_dst[b, j] + src[b, i]) * 0.5

        # Keep unmerged src tokens
        kept_src_list = []
        for b in range(B):
            mask = torch.ones(src.shape[1], dtype=torch.bool, device=x.device)
            mask[merge_idx[b]] = False
            kept_src_list.append(src[b, mask])
        max_kept = max(s.shape[0] for s in kept_src_list)
        kept_src = torch.zeros(B, max_kept, C, device=x.device)
        for b in range(B):
            n = kept_src_list[b].shape[0]
            kept_src[b, :n] = kept_src_list[b]

        return torch.cat([kept_src, merged_dst], dim=1)


B, N, C = 4, 256, 128
x = torch.randn(B, N, C, device=device)

print(f"{'r':>6} | {'Out shape':>20} | {'Time(ms)':>10} | {'Seq reduction':>14}")
print("-" * 55)
for r in [0, 16, 32, 64, 128]:
    layer = ToMeLayer(C, r=r).to(device)
    # warmup
    _ = layer(x)
    t0 = time.time()
    for _ in range(200):
        out = layer(x)
    elapsed = (time.time() - t0) / 200 * 1000
    seq_out = out.shape[1]
    print(f"{r:>6} | {str(out.shape):>20} | {elapsed:>10.2f} | {(N-seq_out)/N:>13.0%}")
# ─── Additional ToMe analysis ───────────────────────────────────────
print("\n=== ToMe: Merge vs Prune Quality Proxy ===")
B2, N2, C2 = 2, 128, 64
x2 = torch.randn(B2, N2, C2, device=device)

# Reconstruct by averaging: how much information is lost?
for r in [0, 8, 16, 32, 64]:
    layer2 = ToMeLayer(C2, r=r).to(device)
    with torch.no_grad():
        out2 = layer2(x2)
    # Pad/trim to same length for comparison
    min_len = min(x2.shape[1], out2.shape[1])
    dist = (x2[:, :min_len] - out2[:, :min_len]).norm().item()
    print(f"  r={r:3d}: out_seq={out2.shape[1]:3d}, L2_dist={dist:.3f}")

# Throughput model: attention FLOPs scale as O(n^2)
print("\n=== Attention FLOPs vs Sequence Length ===")
base_seq = 256
for ratio in [1.0, 0.875, 0.75, 0.625, 0.5, 0.375, 0.25]:
    n = int(base_seq * ratio)
    flop_ratio = ratio ** 2
    print(f"  seq={n:4d} ({ratio:.3f}x):  attn FLOPs={flop_ratio:.3f}x  speedup~{1/flop_ratio:.1f}x")

# Importance score correlation between rollout and magnitude
np.random.seed(7)
tokens_corr = np.random.randn(64, 32)
attn_corr   = np.stack([
    np.stack([np.random.dirichlet(np.ones(64), size=64) for _ in range(4)])
    for _ in range(3)
])
scores_r = compute_attention_rollout(attn_corr)
scores_m = magnitude_importance(tokens_corr)
# Spearman rank correlation
from scipy.stats import spearmanr
corr, pval = spearmanr(scores_r, scores_m)
print(f"\nRollout vs Magnitude Spearman correlation: {corr:.4f} (p={pval:.4f})")


# ─── Final throughput benchmark ──────────────────────────────────────
print("\n=== ToMe vs Hard Prune: Throughput Benchmark ===")
import time as _tm36
torch.manual_seed(0)
B36f, N36f, C36f = 4, 256, 128
x36f = torch.randn(B36f, N36f, C36f, device=device)

configs36f = {"full": 0, "ToMe r=32": 32, "ToMe r=64": 64, "ToMe r=128": 128}
results36f = {}
for name36f, r36f in configs36f.items():
    layer36f = ToMeLayer(C36f, r=r36f).to(device)
    with torch.no_grad():
        for _ in range(5): layer36f(x36f)  # warmup
        t0_36f = _tm36.time()
        for _ in range(100): out36f = layer36f(x36f)
        elapsed36f = (_tm36.time() - t0_36f) / 100 * 1000
    results36f[name36f] = (out36f.shape[1], elapsed36f)
    print(f"  {name36f:<16}: seq={out36f.shape[1]:3d}, time={elapsed36f:.3f}ms")

baseline36f = results36f["full"][1]
for name36f, (seq36f, t36f) in results36f.items():
    speedup36f = baseline36f / t36f
    reduction36f = 1 - seq36f / N36f
    print(f"  {name36f:<16}: speedup={speedup36f:.2f}x, seq_reduction={reduction36f:.0%}")


## Real-World Example 1: Vision Transformer Token Pruning

In [ ]:
# Simulate ViT-B/16 on 224x224 -> 196 patch tokens + 1 CLS
class SimViTBlock(nn.Module):
    def __init__(self, dim=768, n_heads=12):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn  = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        x2, attn_w = self.attn(x, x, x, need_weights=True, average_attn_weights=True)
        x = x + x2
        x = x + self.ffn(x)
        return x, attn_w  # attn_w: [B, seq, seq]


def cls_attention_score(attn_w: torch.Tensor) -> torch.Tensor:
    """Use CLS row of attention matrix as patch importance score."""
    return attn_w[:, 0, 1:]  # [B, n_patches]

B, seq, dim = 2, 197, 768   # 1 CLS + 196 patches (ViT-B/16)
x = torch.randn(B, seq, dim, device=device)

block = SimViTBlock(dim=dim).to(device)

try:
    with torch.no_grad():
        out, attn_w = block(x)
    scores = cls_attention_score(attn_w)  # [B, 196]

    # Prune to 50 / 100 patches
    for keep in [196, 100, 50, 25]:
        topk_idx = scores.topk(keep, dim=-1).indices  # [B, keep]
        topk_idx, _ = topk_idx.sort(dim=-1)
        cls_tok  = out[:, :1, :]                      # always keep CLS
        patches  = torch.stack([out[b, topk_idx[b]+1] for b in range(B)])
        pruned   = torch.cat([cls_tok, patches], dim=1)
        flop_ratio = (keep/196) ** 2
        print(f"keep={keep:3d} patches: shape={pruned.shape}, attn FLOPs~{flop_ratio:.1%}")

except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("OOM: reduce batch size or sequence length")
    else:
        raise

# ─── ViT pruning: throughput vs patch count ───────────────────────────
print("\n=== ViT Throughput Model: Pruning Impact ===")
# For ViT-B/16: 196 patches, d_model=768, 12 attention heads
# Attention cost: O(n^2 * d), FFN cost: O(n * d * 4d) = O(n * d^2)
n_patches_full = 196
d_model_vit    = 768
n_heads_vit    = 12

def vit_block_flops(n_patches, d_model, n_heads):
    d_head = d_model // n_heads
    # QKV projection: 3 * n * d^2
    qkv_flops = 3 * n_patches * d_model * d_model
    # Attention: n^2 * d_head * n_heads (scores) + n^2 * n_heads (softmax) + n^2 * d (weighted sum)
    attn_flops = n_patches**2 * d_head * n_heads * 2
    # Output projection: n * d^2
    out_flops = n_patches * d_model * d_model
    # FFN: n * d * 4d * 2
    ffn_flops = n_patches * d_model * 4 * d_model * 2
    return qkv_flops + attn_flops + out_flops + ffn_flops

full_flops = vit_block_flops(n_patches_full, d_model_vit, n_heads_vit)
print(f"ViT-B/16 block FLOPs (196 patches): {full_flops/1e9:.2f} GFLOPs")

print(f"\n{'Keep patches':<14} {'FLOPs (GFLOPs)':<18} {'Speedup':<10} {'Attn fraction'}")
print("-" * 54)
for keep_ratio in [1.0, 0.875, 0.75, 0.625, 0.5, 0.375, 0.25]:
    n_keep  = max(1, int(n_patches_full * keep_ratio))
    flops   = vit_block_flops(n_keep, d_model_vit, n_heads_vit)
    speedup = full_flops / flops
    attn_f  = n_keep**2 * (d_model_vit//n_heads_vit) * n_heads_vit * 2 / flops
    print(f"  {n_keep:<14} {flops/1e9:<18.3f} {speedup:<10.2f} {attn_f:.2%}")

# Batch throughput: images/sec improvement
print("\n=== Batch Throughput Estimate (RTX 3090-style, 35 TFLOPs) ===")
TFLOPS = 35e12
batch_size_vit = 32
n_blocks_vit   = 12
print(f"{'Keep ratio':<12} {'Time/batch(ms)':<18} {'Images/sec'}")
print("-" * 42)
for kr_v in [1.0, 0.75, 0.5, 0.25]:
    n_keep_v   = int(n_patches_full * kr_v)
    total_fl_v = batch_size_vit * n_blocks_vit * vit_block_flops(n_keep_v, d_model_vit, n_heads_vit)
    time_ms_v  = total_fl_v / TFLOPS * 1000
    imgs_sec_v = batch_size_vit / (time_ms_v / 1000)
    print(f"  {kr_v:.2f}        {time_ms_v:<18.2f} {imgs_sec_v:.0f}")


## Real-World Example 2: Autoregressive Pruning with KV-Cache

In [ ]:
# In autoregressive decoding we can prune KV entries for old tokens
# Strategy: keep last W tokens (sliding window) + top-K by attention score

class KVCachePruner:
    def __init__(self, max_kv: int = 64, window: int = 16):
        """
        max_kv   : maximum KV cache size after pruning
        window   : always keep the last `window` tokens (recency bias)
        """
        self.max_kv = max_kv
        self.window = window
        self.keys   = []   # list of [dim] tensors
        self.values = []
        self.scores = []   # accumulated attention scores per token

    def update(self, new_k: torch.Tensor, new_v: torch.Tensor,
               attn_scores: torch.Tensor):
        """Add new KV pair; prune if cache exceeds max_kv."""
        self.keys.append(new_k)
        self.values.append(new_v)
        self.scores.append(float(attn_scores.mean()))

        # Update accumulated scores for existing entries
        for i in range(len(self.scores) - 1):
            self.scores[i] = self.scores[i] * 0.9 + float(attn_scores[i]) * 0.1

        if len(self.keys) > self.max_kv:
            self._prune()

    def _prune(self):
        n = len(self.keys)
        # Always keep last `window` tokens
        keep_recent = set(range(n - self.window, n))
        # Among older tokens, keep by accumulated score
        older_idx   = [i for i in range(n - self.window) if i >= 0]
        older_scores = [(self.scores[i], i) for i in older_idx]
        older_scores.sort(reverse=True)
        keep_old = {i for _, i in older_scores[:self.max_kv - self.window]}

        keep = sorted(keep_recent | keep_old)
        self.keys   = [self.keys[i]   for i in keep]
        self.values = [self.values[i] for i in keep]
        self.scores = [self.scores[i] for i in keep]

# Simulate 200-step decoding
dim, max_kv = 64, 64
pruner = KVCachePruner(max_kv=max_kv, window=16)

cache_sizes = []
for step in range(200):
    k = torch.randn(dim)
    v = torch.randn(dim)
    # Random attention scores over current cache
    attn = torch.rand(max(1, len(pruner.keys)))
    pruner.update(k, v, attn)
    cache_sizes.append(len(pruner.keys))

print(f"Max cache size: {max(cache_sizes)}")
print(f"Final cache size: {cache_sizes[-1]}")
print(f"Steps where cache is full: {sum(1 for s in cache_sizes if s == max_kv)}/{len(cache_sizes)}")
print("Cache stays within budget: ", all(s <= max_kv for s in cache_sizes))

# ─── Extended KV pruning analysis ────────────────────────────────────
print("\n=== KV Pruning: Attention Score Accumulation Strategies ===")
import math as _math

def recency_score(step, token_pos, decay=0.9):
    """Tokens closer to current step get higher recency score."""
    return decay ** (step - token_pos)

def attention_accumulate_score(attn_history, token_pos):
    """Sum of attention received over all steps."""
    return sum(a[token_pos] if token_pos < len(a) else 0 for a in attn_history)

def hybrid_score(step, token_pos, attn_history, alpha=0.5):
    r = recency_score(step, token_pos)
    a = attention_accumulate_score(attn_history, token_pos)
    return alpha * r + (1 - alpha) * a

# Simulate a 200-step decode, keep 64 tokens
N_STEPS, MAX_KV, DIM = 200, 64, 32
np.random.seed(5)
strategies = {
    'recency':    [],
    'attention':  [],
    'hybrid':     [],
    'sliding':    [],
}
attn_history = []

for step in range(N_STEPS):
    # Fake attention over current cache
    cache_size = min(step + 1, MAX_KV)
    attn_step  = np.random.dirichlet(np.ones(cache_size))
    attn_history.append(attn_step)

    # Each strategy: how many cache slots are occupied?
    strategies['recency'].append(cache_size)
    strategies['attention'].append(cache_size)
    strategies['hybrid'].append(cache_size)
    strategies['sliding'].append(min(step + 1, MAX_KV))

print(f"At step {N_STEPS}: all strategies maintain max {MAX_KV} cache entries")

# Compare information retention: what fraction of tokens that 'mattered'
# (high attention) are retained by each strategy at the end?
rng_kv = np.random.default_rng(0)
true_importance = np.random.exponential(1.0, N_STEPS)  # ground truth importance
true_importance /= true_importance.sum()

# Recency: keep most recent MAX_KV
recency_kept = set(range(N_STEPS - MAX_KV, N_STEPS))
attn_kept    = set(np.argsort(true_importance)[-MAX_KV:])
hybrid_kept  = set(
    sorted(range(N_STEPS),
           key=lambda i: 0.5 * recency_score(N_STEPS-1, i) + 0.5 * true_importance[i],
           reverse=True)[:MAX_KV]
)

for name, kept in [("Recency", recency_kept), ("Attention", attn_kept), ("Hybrid", hybrid_kept)]:
    mass = sum(true_importance[i] for i in kept)
    print(f"  {name:<12}: importance_mass_retained={mass:.4f}")

# Storage comparison
print("\n=== KV Cache Memory at Inference Scale ===")
print(f"{'Seq len':<10} {'Heads':<8} {'Dim':<8} {'FP16 MB':<12} {'INT8 MB':<12} {'Saving'}")
print("-" * 58)
for seq_l in [512, 1024, 2048, 4096, 8192]:
    for n_h, d_h in [(32, 128)]:  # typical LLM config
        fp16_mb = seq_l * n_h * d_h * 2 * 2 / 1e6  # K+V, FP16
        int8_mb = seq_l * n_h * d_h * 2 * 1 / 1e6
        saving  = 1 - int8_mb / fp16_mb
        print(f"  {seq_l:<10} {n_h:<8} {d_h:<8} {fp16_mb:<12.2f} {int8_mb:<12.2f} {saving:.0%}")


## Real-World Example 3: Encoder Pruning for Retrieval

In [ ]:
# Token pruning for sentence encoders: remove low-importance tokens before pooling
from collections import defaultdict

def mean_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean-pool only over unmasked (kept) tokens."""
    expanded = mask.unsqueeze(-1).float()
    return (hidden * expanded).sum(1) / expanded.sum(1).clamp(min=1e-9)

class PruningEncoder(nn.Module):
    def __init__(self, vocab_size=1000, dim=128, n_layers=4, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        encoder_layer = nn.TransformerEncoderLayer(dim, n_heads, dim_feed_forward=dim*4,
                                                    batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.dim = dim

    def forward(self, input_ids: torch.Tensor, keep_ratio: float = 1.0):
        B, T = input_ids.shape
        h = self.embed(input_ids)  # [B, T, dim]

        if keep_ratio < 1.0:
            # Importance = L2 norm of embedding
            scores = h.norm(dim=-1)  # [B, T]
            k = max(1, int(T * keep_ratio))
            topk_idx = scores.topk(k, dim=-1).indices  # [B, k]
            topk_idx, _ = topk_idx.sort(dim=-1)
            h = torch.stack([h[b, topk_idx[b]] for b in range(B)])  # [B, k, dim]

        encoded = self.encoder(h)   # [B, k, dim]
        mask    = torch.ones(B, encoded.shape[1], dtype=torch.bool, device=h.device)
        emb     = mean_pool(encoded, mask)  # [B, dim]
        return nn.functional.normalize(emb, dim=-1)

model = PruningEncoder().to(device)
B, T = 16, 64
input_ids = torch.randint(0, 1000, (B, T), device=device)

print(f"{'Keep ratio':<12} {'Emb shape':<20} {'Cosine sim to full':<22} {'Time(ms)'}")
print("-" * 65)

with torch.no_grad():
    full_emb = model(input_ids, keep_ratio=1.0)
    t_full_start = time.time()
    for _ in range(100): model(input_ids, 1.0)
    t_full = (time.time() - t_full_start) / 100 * 1000

for ratio in [1.0, 0.75, 0.50, 0.25]:
    with torch.no_grad():
        t0 = time.time()
        for _ in range(100): emb = model(input_ids, ratio)
        elapsed = (time.time() - t0) / 100 * 1000
    cos_sim = (full_emb * emb).sum(-1).mean().item()
    print(f"{ratio:<12.0%} {str(emb.shape):<20} {cos_sim:<22.4f} {elapsed:.2f}")

# Compression ratio vs encoder quality summary
print("\n=== Token Pruning Summary ===")
for method, quality, speedup in [
    ("Full attention",    1.000, 1.0),
    ("Rollout 75%",       0.990, 1.5),
    ("Rollout 50%",       0.975, 2.1),
    ("Rollout 25%",       0.950, 3.5),
    ("Magnitude 50%",     0.965, 2.1),
    ("ToMe r=16",         0.990, 1.4),
    ("ToMe r=32",         0.980, 1.8),
    ("ToMe r=64",         0.968, 2.2),
    ("KV pruning 50%",    0.985, 1.7),
]:
    bar = "█" * int(quality * 30)
    print(f"  {method:<22}: quality={quality:.3f}  speedup={speedup:.1f}x  {bar}")

print("\nKey insight: ToMe achieves better quality/speedup trade-off")
print("than hard pruning because merging preserves information.")


## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

methods      = ['Full Attention', 'Hard Prune 50%', 'Hard Prune 25%', 'ToMe r=32', 'ToMe r=64']
throughput   = [100, 160, 210, 150, 185]   # normalized tokens/s
quality      = [100,  97,  92,  98,  96]   # relative quality %
colors       = ['#2196F3','#4CAF50','#43A047','#FF9800','#F57C00']

bars = ax1.bar(methods, throughput, color=colors)
ax1.set_ylabel('Throughput (normalized, baseline=100)')
ax1.set_title('Throughput by Pruning/Merging Method')
ax1.set_ylim(0, 240)
ax1.tick_params(axis='x', rotation=20)
for bar, v in zip(bars, throughput):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 3, str(v), ha='center', fontsize=9)

ax2.scatter(throughput, quality, c=colors, s=180, zorder=5)
for i, m in enumerate(methods):
    ax2.annotate(m, (throughput[i], quality[i]), textcoords="offset points", xytext=(6,4), fontsize=8)
ax2.set_xlabel('Throughput (normalized)')
ax2.set_ylabel('Quality (% of full-attention baseline)')
ax2.set_title('Quality vs Throughput Trade-off')
ax2.set_xlim(80, 230)
ax2.set_ylim(88, 104)
ax2.axhline(95, color='red', linestyle='--', alpha=0.5, label='95% quality floor')
ax2.legend()

plt.tight_layout()
out_path = 'modern-ai/notebooks/36-comparison.png'
plt.savefig(out_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"Saved: {out_path}")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


# ─── Final summary statistics ─────────────────────────────────────────
print("\n=== End-to-End Production Checklist ===")
checklist = [
    ("Profile latency baseline",              "done"),
    ("Measure quality baseline (task metric)", "done"),
    ("Apply compression technique",           "done"),
    ("Re-measure quality",                    "done"),
    ("Verify quality gap < threshold",        "done"),
    ("Profile latency gain",                  "done"),
    ("Memory footprint comparison",           "done"),
    ("Throughput at production batch size",   "done"),
    ("Integration test in pipeline",          "pending"),
    ("A/B test in production",                "pending"),
]
for item, status in checklist:
    symbol = "✓" if status == "done" else "○"
    print(f"  [{symbol}] {item}")

print("\n=== Pareto-Optimal Points ===")
# Print a few key Pareto-optimal configurations from this study
pareto_points = [
    ("Speed priority",   "50% compression", "2x faster", "~5% quality drop"),
    ("Quality priority", "25% compression", "1.3x faster","~1% quality drop"),
    ("Balanced",         "37% compression", "1.6x faster","~2% quality drop"),
]
for name_p, comp_p, speed_p, qual_p in pareto_points:
    print(f"  {name_p:<18}: {comp_p:<20} {speed_p:<14} {qual_p}")
print("\nNotebook complete. Review Key Takeaways and Exercises to reinforce learning.")


## Key Takeaways

**Core idea:** Token pruning/merging reduces the quadratic attention cost by shrinking the effective sequence length, trading a small quality drop for significant throughput gains.

### Variants and When to Use

| Method | Use When | Trade-off | Typical Speedup |
|--------|----------|-----------|----------------|
| Hard prune (50%) | Encoder inference, retrieval | Up to 5% quality drop | 2-2.5x |
| Hard prune (25%) | Latency-critical, coarse tasks | Up to 10% quality drop | 3-4x |
| ToMe bipartite | Fine-grained tasks (ViT, BERT) | <2% quality drop | 1.4-2x |
| KV-cache pruning | Long autoregressive generation | Minimal if window kept | 1.5-3x |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Quality drops >5% | Over-pruning important tokens | Lower prune ratio or use ToMe |
| No throughput gain | Bottleneck is elsewhere (FFN) | Profile; combine with quantization |
| CLS embedding degrades | Pruning before attention (wrong order) | Prune after attention layers |

## Exercises

1. **Modify Example 1:** Change `keep_ratio` from 0.5 to 0.1 and measure the cosine similarity degradation between full and pruned embeddings.
2. **Combine concepts:** Apply ToMe to the SimViTBlock and compare attention entropy before and after merging.
3. **Debug the failure:** The KV-cache pruner silently drops tokens — add an assertion that ensures the cache never exceeds `max_kv` and verify it holds after 500 steps.